# Coach changes in Italian volleyball: from matches to coach changes

Shows how the coach spells, the mid-season coach changes and the coverage table in `volley_coaches_clean.zip` are built
from the match files, and checks that the rebuilt files are identical to those in the zip.

**Input:** `matches.csv` (one row per match) and `team_matches.csv` (one row per team and match), both in `volley_coaches_clean.zip`.
Every row lists the match reports it comes from (`links`) and the head coaches as printed in them (`coach_H_raw`, `coach_A_raw`,
`coach_raw`), so each match and each coach can be checked against its source.

**Output:** `team_matches_regular_with_perf.csv`, `coach_spells.csv`, `coach_changes.csv`, `coach_returns.csv` and `coverage.csv`,
saved in `volley_coaches_rebuilt.zip`.

**How to run:** in Google Colab, Runtime, then Run all. Nothing needs editing: the notebook uses `volley_coaches_clean.zip` if it is
in the working folder and otherwise downloads it from the repository.

## 1. Settings and data

In [ ]:
ZIP_NAME = "volley_coaches_clean.zip"
REPO_ZIP = "https://raw.githubusercontent.com/manueleleonelli/Volley_Coaches/main/volley_coaches_clean.zip"
MIN_MATCHES = 3   # matches a coach must lead to count as outgoing or incoming coach
MIN_LEFT = 3      # matches that must remain in the regular season after the change

import os, sys, zipfile, urllib.request
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
WORK = "/content/coach_build" if IN_COLAB else os.path.abspath("coach_build")
DATA, OUT = os.path.join(WORK, "data"), os.path.join(WORK, "out")
os.makedirs(DATA, exist_ok=True)
os.makedirs(OUT, exist_ok=True)
zip_path = ZIP_NAME if os.path.exists(ZIP_NAME) else os.path.join(WORK, ZIP_NAME)
if not os.path.exists(zip_path):
    print("Downloading", REPO_ZIP)
    urllib.request.urlretrieve(REPO_ZIP, zip_path)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(DATA)

matches = pd.read_csv(os.path.join(DATA, "matches.csv"), low_memory=False)
tm = pd.read_csv(os.path.join(DATA, "team_matches.csv"), low_memory=False)
print("Matches:", len(matches), "| team-matches:", len(tm))
print(matches["phase"].value_counts())

## 2. Regular-season matches

Only regular-season matches are used. Within each team-season they are numbered by date, `t` = 1, 2, ...

In [ ]:
reg = tm[tm["phase"] == "regular"].copy()
reg = reg.sort_values(["league", "season", "team", "date", "cid"], kind="stable")
reg["t"] = reg.groupby(["league", "season", "team"]).cumcount() + 1
print("Regular-season team-matches:", len(reg), "| matches:", reg["cid"].nunique(),
      "| team-seasons:", reg.groupby(["league", "season", "team"]).ngroups,
      "| without a head coach in the reports:", int(reg["coach"].isna().sum()))

## 3. Coach spells

A spell collects all the regular-season matches in which a head coach led a team in a season: the number of matches,
the first and the last match.

In [ ]:
spells = (reg.dropna(subset=["coach"]).groupby(["league", "season", "team", "coach"])
          .agg(coach_name=("coach_name", "first"), matches=("t", "size"), t_first=("t", "min"), t_last=("t", "max"),
               date_first=("date", "min"), date_last=("date", "max")).reset_index())
print("Spells:", len(spells), "| with at least", MIN_MATCHES, "matches:", int((spells["matches"] >= MIN_MATCHES).sum()))

## 4. Mid-season coach changes

For every team-season, a coach change is recorded at the last match of an outgoing coach when

1. the outgoing coach led at least `MIN_MATCHES` matches of the team in the season, and does not appear after that match;
2. at least `MIN_LEFT` regular-season matches of the team remain;
3. the incoming coach, the coach who led most of the remaining matches, led at least `MIN_MATCHES` of them;
4. the incoming coach did not lead the team before the first match of the outgoing coach. Otherwise the outgoing coach was a
   stand-in and the original coach returned: these cases are listed in `coach_returns.csv` and are not changes;
5. the incoming coach did not lead more than `MIN_MATCHES` matches before the last match of the outgoing coach, which would mean
   that the two coaches alternated rather than one replacing the other.

The first coach after the change (`first_successor`) differs from the incoming coach when a caretaker led the team in between.
`overlap` marks the changes in which the incoming coach appears in a report before the last match of the outgoing coach,
as a stand-in or by mistake. `season_opener` marks the first change of a team-season.

In [ ]:
changes, returns = [], []
for key, d in reg.groupby(["league", "season", "team"]):
    d = d.set_index("t")
    n = len(d)
    sp = spells[(spells["league"] == key[0]) & (spells["season"] == key[1]) & (spells["team"] == key[2])]
    for s in sp[sp["matches"] >= MIN_MATCHES].itertuples():
        if n - s.t_last < MIN_LEFT:                                   # rule 2
            continue
        after = d.loc[s.t_last + 1:]
        counts = after["coach"].dropna().value_counts()
        if counts.empty or counts.iloc[0] < MIN_MATCHES:              # rule 3
            continue
        new = counts.index[0]
        before = d.loc[:s.t_last - 1, "coach"]
        if sp.loc[sp["coach"] == new, "t_first"].iloc[0] < s.t_first:  # rule 4
            returns.append(dict(league=key[0], season=key[1], team=key[2], out_coach=s.coach, in_coach=new, t_last=s.t_last))
            continue
        if (before == new).sum() > MIN_MATCHES:                        # rule 5
            continue
        first = after.dropna(subset=["coach"]).iloc[0]
        changes.append(dict(
            league=key[0], sex=d["sex"].iloc[0], season=key[1], team=key[2],
            out_coach=s.coach, out_name=s.coach_name, in_coach=new, in_name=d.loc[d["coach"] == new, "coach_name"].iloc[0],
            first_successor=first["coach_name"], t_last=s.t_last, date_last=d.at[s.t_last, "date"],
            date_next=after["date"].iloc[0], matches_out=s.matches, matches_after=n - s.t_last, team_matches=n,
            overlap=bool((before == new).any()),
            link_last=str(d.at[s.t_last, "links"]).split()[0], link_next=str(after["links"].iloc[0]).split()[0]))
coach_changes = pd.DataFrame(changes)
coach_changes["season_opener"] = (coach_changes["t_last"]
                                  == coach_changes.groupby(["league", "season", "team"])["t_last"].transform("min"))
coach_returns = pd.DataFrame(returns)
print("Coach changes:", len(coach_changes), "in", coach_changes.groupby(["league", "season", "team"]).ngroups, "team-seasons",
      "| first of a season:", int(coach_changes["season_opener"].sum()),
      "| caretaker first:", int((coach_changes["first_successor"] != coach_changes["in_name"]).sum()),
      "| overlap:", int(coach_changes["overlap"].sum()), "| returns (not changes):", len(coach_returns))
print(coach_changes.groupby("league").size())

## 5. Coverage

Within each league-season, the teams that play each other form the groups of the regular season (one group, or two in some
seasons of the lower leagues). A complete double round robin has $g(g-1)$ matches in a group of $g$ teams. `official_share` is the
percentage of matches listed in the league's official report of a regular-season round.

In [ ]:
rows = []
for (lg, se), g in matches[matches["phase"] == "regular"].groupby(["league", "season"]):
    parent = {}

    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for a, b in zip(g["H"], g["A"]):
        parent[find(a)] = find(b)
    comp = {}
    for team in set(g["H"]) | set(g["A"]):
        comp.setdefault(find(team), set()).add(team)
    sizes = sorted((len(c) for c in comp.values()), reverse=True)
    expected = sum(s * (s - 1) for s in sizes)
    rows.append(dict(league=lg, season=se, groups=len(sizes), group_sizes=" ".join(map(str, sizes)), matches=len(g),
                     expected=expected, official_share=round(100 * g["official"].mean(), 1),
                     pct=round(100 * len(g) / expected, 1)))
coverage = pd.DataFrame(rows)
print("Coverage:", round(100 * coverage["matches"].sum() / coverage["expected"].sum(), 1), "% of a complete double round robin")

## 6. Check against the files in the zip, and save

In [ ]:
CHECK = {"team_matches_regular_with_perf.csv": (reg, ["league", "season", "team", "t"]),
         "coach_spells.csv": (spells, ["league", "season", "team", "coach"]),
         "coach_changes.csv": (coach_changes, ["league", "season", "team", "t_last"]),
         "coach_returns.csv": (coach_returns, ["league", "season", "team", "t_last"]),
         "coverage.csv": (coverage, ["league", "season"])}
for name, (df, keys) in CHECK.items():
    ref = pd.read_csv(os.path.join(DATA, name), low_memory=False)
    a = df[ref.columns].sort_values(keys).reset_index(drop=True).astype(str)
    b = ref.sort_values(keys).reset_index(drop=True).astype(str)
    print(f"{name}: {len(df)} rows,", "identical to the zip" if a.equals(b) else "DIFFERENT from the zip")
    df.to_csv(os.path.join(OUT, name), index=False)
zpath = os.path.join(WORK, "volley_coaches_rebuilt.zip")
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for name in CHECK:
        z.write(os.path.join(OUT, name), name)
print("Saved", zpath)